[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/megacare-dev/agentic_rag_workshop/blob/main/th/knowledge_graph/knowledge_graph_rag.ipynb)

# 🕸️ Knowledge Graph RAG
## Agentic RAG Workshop — Bonus: GraphRAG

---

### 🎯 จุดประสงค์การเรียนรู้

เมื่อจบ notebook นี้ผู้เรียนสามารถ:
1. **อธิบาย Knowledge Graph** — Entity, Relation, Triple คืออะไร
2. **สกัด Entity & Relation** จากข้อความภาษาไทยด้วย Gemini
3. **สร้าง Knowledge Graph** ด้วย NetworkX และ visualize ได้
4. **Query Knowledge Graph** เพื่อค้นหาความสัมพันธ์
5. **สร้าง GraphRAG Pipeline** — ค้นหาจาก Graph แล้วให้ LLM ตอบ
6. **เปรียบเทียบ** Vector RAG vs GraphRAG ว่าเหมาะกับงานแบบไหน

### 📦 สิ่งที่ต้องการ
- Google Account + Gemini API Key
- ไม่ต้องติดตั้งอะไรล่วงหน้า — ใช้ Google Colab

---

### 🗺️ Pipeline วันนี้

```
📄 ข้อความ
    │
    ▼
🤖 Gemini: สกัด Entity & Relation
    │
    ▼
🕸️ Knowledge Graph (NetworkX)
    │
    ├──► 🔍 Query: ค้นหาความสัมพันธ์
    │
    └──► 🤖 GraphRAG: ส่ง context ให้ LLM ตอบ
```

---
## 📦 Section 0: ติดตั้ง Dependencies

In [ ]:
%%time
import importlib.util, subprocess, sys

def _pip_install(pkg_spec, import_name=None):
    pkg = pkg_spec.split('>=')[0].split('==')[0].split('[')[0].strip()
    imp = import_name or {
        'google-genai': 'google.genai',
    }.get(pkg, pkg.replace('-', '_'))
    try:
        spec = importlib.util.find_spec(imp)
    except ModuleNotFoundError:
        spec = None
    if spec is not None:
        print(f'  ⏭️  {pkg}: skipped')
        return
    print(f'  📦 {pkg}: installing...', end='', flush=True)
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg_spec],
                       capture_output=True, text=True)
    print(f'\r  ✅ {pkg}: done' if r.returncode == 0 else f'\r  ❌ {pkg}: failed')

for _pkg in ['google-genai', 'networkx', 'matplotlib']:
    _pip_install(_pkg)

print('\n✅ พร้อมใช้งาน!')

### 🔑 ตั้งค่า Gemini API Key

In [ ]:
import os
import google.genai as genai

try:
    from google.colab import userdata
    GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
    print('✅ โหลด API Key จาก Colab Secrets')
except Exception:
    GEMINI_API_KEY = input('วาง Gemini API Key แล้วกด Enter: ').strip()

client = genai.Client(api_key=GEMINI_API_KEY)
print('✅ Gemini client พร้อมใช้งาน')

---
## 📖 Section 1: Knowledge Graph คืออะไร?

Knowledge Graph คือโครงสร้างข้อมูลที่แทนความรู้ในรูปแบบ **กราฟ** ประกอบด้วย:

| องค์ประกอบ | คืออะไร | ตัวอย่าง |
|---|---|---|
| **Entity (Node)** | สิ่งที่เราพูดถึง | `Beryl8`, `กรุงเทพ`, `Python` |
| **Relation (Edge)** | ความสัมพันธ์ระหว่าง Entity | `ก่อตั้งที่`, `ใช้ภาษา`, `เป็นส่วนหนึ่งของ` |
| **Triple** | Entity → Relation → Entity | `Beryl8 → ก่อตั้งที่ → กรุงเทพ` |

### 🔍 ทำไม Knowledge Graph ถึงดีกว่า Vector Search ในบางกรณี?

```
❓ คำถาม: "บริษัทที่ตั้งอยู่ในกรุงเทพและใช้ Python มีอะไรบ้าง?"

Vector RAG: หา chunk ที่ semantic คล้ายกับคำถาม
  → อาจพลาดถ้า chunk ไม่มีคำตรงๆ

GraphRAG:   traversal graph จาก node 'กรุงเทพ' → บริษัทที่ตั้งอยู่ที่นั่น
           + traversal จาก node 'Python' → บริษัทที่ใช้
           → intersection = คำตอบที่แม่นยำ
```

**GraphRAG เก่งเรื่อง:** multi-hop reasoning, ความสัมพันธ์ซับซ้อน, คำถาม "ใครเชื่อมกับใคร"

**Vector RAG เก่งเรื่อง:** semantic similarity, คำถามทั่วไป, ข้อมูลที่ไม่มีโครงสร้างชัดเจน

---
## 🤖 Section 2: สกัด Entity & Relation ด้วย Gemini

เราจะใช้ Gemini อ่านข้อความแล้วสกัด **Triple** ออกมาในรูปแบบ JSON

In [ ]:
# ข้อมูลตัวอย่าง — เกี่ยวกับบริษัท Tech ในไทย
SAMPLE_TEXTS = [
    """
    Beryl8 เป็นบริษัท Data & AI Consulting ก่อตั้งอยู่ในกรุงเทพมหานคร
    บริษัทเชี่ยวชาญด้าน Machine Learning, Data Engineering และ Generative AI
    Beryl8 เป็น Google Cloud Partner และ Databricks Partner
    ทีม Data Engineering ของ Beryl8 ใช้ภาษา Python และ SQL เป็นหลัก
    """,
    """
    Agentic RAG เป็นเทคนิคที่รวม AI Agent เข้ากับ RAG Pipeline
    RAG ย่อมาจาก Retrieval-Augmented Generation
    Agentic RAG ใช้ Vector Database เช่น Qdrant สำหรับเก็บ Embedding
    Google ADK เป็น Framework สำหรับสร้าง AI Agent ที่พัฒนาโดย Google
    Gemini เป็น LLM ที่ Google พัฒนาและใช้ใน Google ADK
    """,
    """
    Knowledge Graph เป็นโครงสร้างข้อมูลแบบกราฟ
    Knowledge Graph ประกอบด้วย Entity และ Relation
    GraphRAG ใช้ Knowledge Graph ร่วมกับ LLM เพื่อตอบคำถาม
    Neo4j และ NetworkX เป็นเครื่องมือยอดนิยมสำหรับสร้าง Knowledge Graph
    """
]

In [ ]:
import json

EXTRACT_PROMPT = """
อ่านข้อความต่อไปนี้แล้วสกัด Triple (Entity, Relation, Entity) ออกมา

กฎ:
- Entity: ชื่อเฉพาะ, เทคโนโลยี, บริษัท, แนวคิด (ไม่ใช่คำกริยาทั่วไป)
- Relation: กริยาที่แสดงความสัมพันธ์ สั้น กระชับ
- ส่งกลับเป็น JSON array เท่านั้น ไม่มีข้อความอื่น

ตัวอย่าง output:
[
  {"subject": "Beryl8", "relation": "ก่อตั้งที่", "object": "กรุงเทพ"},
  {"subject": "Python", "relation": "เป็นภาษาโปรแกรม", "object": "ภาษาโปรแกรม"}
]

ข้อความ:
{text}

JSON output:
"""

def extract_triples(text: str) -> list[dict]:
    """ใช้ Gemini สกัด Triple จากข้อความ"""
    response = client.models.generate_content(
        model='gemini-2.0-flash',
        contents=EXTRACT_PROMPT.format(text=text)
    )
    raw = response.text.strip()
    # ลบ markdown code block ถ้ามี
    if raw.startswith('```'):
        raw = raw.split('```')[1]
        if raw.lower().startswith('json'):
            raw = raw[4:]
    raw = raw.strip()
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        print(f'⚠️ JSON parse error — Gemini ส่งกลับรูปแบบไม่ถูกต้อง:')
        print(raw[:300])
        return []

# ทดสอบกับข้อความแรก
sample_triples = extract_triples(SAMPLE_TEXTS[0])
print(f'สกัดได้ {len(sample_triples)} triples:')
for t in sample_triples:
    print(f"  {t['subject']} --[{t['relation']}]--> {t['object']}")

In [ ]:
# สกัด Triple จากทุก text
all_triples = []
for i, text in enumerate(SAMPLE_TEXTS):
    print(f'📄 กำลังสกัดจาก text {i+1}/{len(SAMPLE_TEXTS)}...')
    triples = extract_triples(text)
    all_triples.extend(triples)
    print(f'   ได้ {len(triples)} triples')

print(f'\n✅ รวมทั้งหมด: {len(all_triples)} triples')

---
## 🕸️ Section 3: สร้าง Knowledge Graph ด้วย NetworkX

In [ ]:
import subprocess, matplotlib.font_manager as fm
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib

# ติดตั้ง Noto Sans Thai เพื่อ render ภาษาไทยใน graph
result = subprocess.run(['apt-get', 'install', '-y', '-q', 'fonts-noto-cjk'],
                        capture_output=True, text=True)
fm._load_fontmanager(try_read_cache=False)
thai_fonts = [f.name for f in fm.fontManager.ttflist if 'Noto' in f.name]
matplotlib.rcParams['font.family'] = thai_fonts[0] if thai_fonts else 'DejaVu Sans'
print(f'🔤 ใช้ font: {matplotlib.rcParams["font.family"]}')

# สร้าง Directed Graph
G = nx.DiGraph()

for triple in all_triples:
    subj = triple['subject'].strip()
    rel  = triple['relation'].strip()
    obj  = triple['object'].strip()
    if subj and rel and obj:
        G.add_edge(subj, obj, relation=rel)

print(f'📊 Knowledge Graph Stats:')
print(f'   Nodes (Entity): {G.number_of_nodes()}')
print(f'   Edges (Relation): {G.number_of_edges()}')
print(f'\n🔝 Top 10 nodes (by degree):')
top_nodes = sorted(G.degree(), key=lambda x: x[1], reverse=True)[:10]
for node, deg in top_nodes:
    print(f'   {node}: {deg} connections')

In [ ]:
# Visualize Knowledge Graph ด้วย matplotlib
plt.figure(figsize=(14, 10))

# Layout
pos = nx.spring_layout(G, k=2, seed=42)

# Node size ตาม degree
node_sizes = [300 + G.degree(n) * 200 for n in G.nodes()]

# วาด Graph
nx.draw_networkx_nodes(G, pos, node_size=node_sizes,
                       node_color='#4A90D9', alpha=0.8)
nx.draw_networkx_labels(G, pos, font_size=8, font_color='white', font_weight='bold')
nx.draw_networkx_edges(G, pos, edge_color='#888', arrows=True,
                       arrowsize=15, width=1.5, alpha=0.6,
                       connectionstyle='arc3,rad=0.1')

# Edge labels (relation)
edge_labels = nx.get_edge_attributes(G, 'relation')
nx.draw_networkx_edge_labels(G, pos, edge_labels, font_size=6,
                              font_color='#333', label_pos=0.3)

plt.title('🕸️ Knowledge Graph — Agentic RAG Workshop', fontsize=14, pad=20)
plt.axis('off')
plt.tight_layout()
plt.savefig('knowledge_graph.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 บันทึกรูปเป็น knowledge_graph.png')

---
## 🔍 Section 4: Query Knowledge Graph

ค้นหาความสัมพันธ์ใน Graph โดยไม่ต้องใช้ LLM

In [ ]:
def find_neighbors(graph: nx.DiGraph, entity: str, depth: int = 1) -> list[dict]:
    """ค้นหา Entity ที่เชื่อมต่อกับ entity ที่ระบุ"""
    # หา entity ที่ชื่อใกล้เคียง (case-insensitive)
    matches = [n for n in graph.nodes() if entity.lower() in n.lower()]
    if not matches:
        return []

    results = []
    for match in matches:
        # Outgoing edges
        for _, obj, data in graph.out_edges(match, data=True):
            results.append({'subject': match, 'relation': data['relation'], 'object': obj})
        # Incoming edges
        for subj, _, data in graph.in_edges(match, data=True):
            results.append({'subject': subj, 'relation': data['relation'], 'object': match})
    return results

def triples_to_context(triples: list[dict]) -> str:
    """แปลง Triple list เป็น text สำหรับส่งให้ LLM"""
    if not triples:
        return 'ไม่พบข้อมูลที่เกี่ยวข้อง'
    lines = [f"- {t['subject']} {t['relation']} {t['object']}" for t in triples]
    return '\n'.join(lines)

# ทดสอบ query
for query_entity in ['Beryl8', 'RAG', 'Knowledge Graph']:
    triples = find_neighbors(G, query_entity)
    print(f'\n🔍 ค้นหา "{query_entity}" — พบ {len(triples)} relations:')
    for t in triples[:5]:
        print(f"   {t['subject']} --[{t['relation']}]--> {t['object']}")
    if len(triples) > 5:
        print(f'   ... และอีก {len(triples)-5} รายการ')

In [ ]:
def find_path(graph: nx.DiGraph, source: str, target: str) -> list:
    """หา path ระหว่าง 2 entities — multi-hop reasoning"""
    # หา nodes ที่ match
    src_nodes = [n for n in graph.nodes() if source.lower() in n.lower()]
    tgt_nodes = [n for n in graph.nodes() if target.lower() in n.lower()]

    for src in src_nodes:
        for tgt in tgt_nodes:
            try:
                path = nx.shortest_path(graph, src, tgt)
                return path
            except nx.NetworkXNoPath:
                continue
    return []

# ทดสอบ multi-hop: Beryl8 เชื่อมกับ Gemini ยังไง?
print('🔗 Multi-hop reasoning:')
pairs = [('Beryl8', 'Gemini'), ('RAG', 'Google'), ('Agentic RAG', 'Qdrant')]
for src, tgt in pairs:
    path = find_path(G, src, tgt)
    if path:
        print(f'\n  {src} → {tgt}:')
        for i in range(len(path) - 1):
            rel = G[path[i]][path[i+1]]['relation']
            print(f'    {path[i]} --[{rel}]--> {path[i+1]}')
    else:
        print(f'\n  {src} → {tgt}: ไม่พบ path')

---
## 🤖 Section 5: GraphRAG Pipeline

รวม Knowledge Graph + Gemini เพื่อตอบคำถาม:
1. รับคำถาม
2. Gemini สกัด Entity จากคำถาม
3. Query Knowledge Graph หา triples ที่เกี่ยวข้อง
4. ส่ง triples เป็น context ให้ Gemini ตอบ

In [ ]:
def extract_query_entities(question: str) -> list[str]:
    """สกัด Entity จากคำถามด้วย Gemini"""
    prompt = f"""
สกัดชื่อ Entity (บริษัท, เทคโนโลยี, แนวคิด) จากคำถามต่อไปนี้
ส่งกลับเป็น JSON array ของ string เท่านั้น

คำถาม: {question}

JSON output:
"""
    response = client.models.generate_content(
        model='gemini-2.0-flash',
        contents=prompt
    )
    raw = response.text.strip()
    if raw.startswith('```'):
        raw = raw.split('```')[1]
        if raw.lower().startswith('json'):
            raw = raw[4:]
    raw = raw.strip()
    try:
        return json.loads(raw)
    except Exception:
        print(f'⚠️ ไม่สามารถสกัด entity จากคำถาม: {question[:50]}')
        return []


def graphrag_query(question: str, graph: nx.DiGraph, verbose: bool = True) -> str:
    """GraphRAG: ตอบคำถามโดยใช้ Knowledge Graph เป็น context"""

    # 1. สกัด Entity จากคำถาม
    entities = extract_query_entities(question)
    if not entities:
        print('⚠️ ไม่พบ entity จากคำถาม — ลองถามใหม่ให้ระบุชื่อที่ชัดเจนขึ้น')
        return ''
    if verbose:
        print(f'🔍 Entity ที่สกัดได้: {entities}')

    # 2. Query Graph
    relevant_triples = []
    for entity in entities:
        triples = find_neighbors(graph, entity)
        relevant_triples.extend(triples)

    # ลบ duplicate
    seen = set()
    unique_triples = []
    for t in relevant_triples:
        key = (t['subject'], t['relation'], t['object'])
        if key not in seen:
            seen.add(key)
            unique_triples.append(t)

    context = triples_to_context(unique_triples)
    if verbose:
        print(f'📊 พบ {len(unique_triples)} relevant triples')
        print(f'📋 Context:\n{context}\n')

    # 3. ส่งให้ Gemini ตอบ
    answer_prompt = f"""
ใช้ข้อมูลต่อไปนี้ตอบคำถาม ถ้าข้อมูลไม่พอให้บอกว่า "ไม่มีข้อมูลเพียงพอ"

ข้อมูลจาก Knowledge Graph:
{context}

คำถาม: {question}

คำตอบ:
"""
    response = client.models.generate_content(
        model='gemini-2.0-flash',
        contents=answer_prompt
    )
    return response.text.strip()

In [ ]:
# ทดสอบ GraphRAG
questions = [
    'Beryl8 เชี่ยวชาญด้านอะไรบ้าง?',
    'Agentic RAG ใช้เครื่องมืออะไรบ้าง?',
    'Knowledge Graph กับ GraphRAG เกี่ยวข้องกันอย่างไร?',
]

for q in questions:
    print('='*60)
    print(f'❓ คำถาม: {q}')
    print()
    answer = graphrag_query(q, G)
    print(f'💬 คำตอบ: {answer}')
    print()

---
## ⚖️ Section 6: Vector RAG vs GraphRAG — เลือกใช้อะไร?

| | **Vector RAG** | **GraphRAG** |
|---|---|---|
| **เหมาะกับ** | คำถามทั่วไป, semantic search | คำถามเกี่ยวกับความสัมพันธ์ |
| **จุดแข็ง** | เร็ว, ง่าย, ยืดหยุ่น | multi-hop reasoning, precise |
| **จุดอ่อน** | พลาดความสัมพันธ์ที่ซับซ้อน | ต้องสร้าง graph ล่วงหน้า |
| **ตัวอย่างคำถาม** | "RAG คืออะไร?" | "บริษัทที่ใช้ Python และ Google Cloud มีใครบ้าง?" |
| **เครื่องมือ** | Qdrant, Pinecone, ChromaDB | NetworkX, Neo4j, Amazon Neptune |

### 🏆 Hybrid: Vector + Graph = ดีที่สุด

```
คำถาม
  │
  ├──► Vector Search → chunks ที่ semantic คล้าย
  │
  └──► Graph Query  → triples ที่เกี่ยวข้อง
            │
            └──► รวม context → LLM → คำตอบที่ครบถ้วน
```

In [ ]:
# สรุปสถิติ Knowledge Graph ที่สร้าง
print('📊 สรุป Knowledge Graph ของเรา')
print('='*40)
print(f'  Entities:   {G.number_of_nodes()} nodes')
print(f'  Relations:  {G.number_of_edges()} edges')
print(f'  Triples:    {len(all_triples)} triples สกัดจาก {len(SAMPLE_TEXTS)} texts')
print()
print('🔝 Top entities (most connected):')
for node, deg in sorted(G.degree(), key=lambda x: x[1], reverse=True)[:5]:
    in_deg = G.in_degree(node)
    out_deg = G.out_degree(node)
    print(f'  {node:30} (in={in_deg}, out={out_deg})')
print()
print('✅ Knowledge Graph RAG เสร็จสมบูรณ์!')

---
## 🚀 ต่อยอด: ลองกับข้อมูลของคุณเอง

แก้ไข `SAMPLE_TEXTS` ด้านบนให้เป็นข้อมูลที่คุณสนใจ เช่น:
- รายงานประจำปีบริษัท
- เอกสาร specification
- บทความข่าว

แล้วรัน Section 2–5 ใหม่ทั้งหมด — Knowledge Graph จะถูกสร้างจากข้อมูลของคุณโดยอัตโนมัติ

```python
# เพิ่มข้อมูลของคุณที่นี่
MY_TEXTS = [
    """ข้อความของคุณที่นี่...""",
]
```

---
*Agentic RAG Workshop — [megacare-dev/agentic_rag_workshop](https://github.com/megacare-dev/agentic_rag_workshop)*